## **XGBoost + lightgbm**

In [1]:
import zipfile
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import RobustScaler
import xgboost as xgb
import lightgbm as lgb
from scipy.stats import rankdata
import numpy as np
import pandas as pd
from scipy.stats import entropy

TOTAL_ITEMS = 1000
RATING_RANGE = range(6)

def split_datasets(X, y, train_users, val_users):
    # Split interactions dataframe into train, val sets based on user
    X_train = X[X["user"].isin(train_users["user"])]
    X_val = X[X["user"].isin(val_users["user"])]

    # Split label dataframe into train, val sets based on user
    y_train = y[y["user"].isin(train_users["user"])]
    y_val = y[y["user"].isin(val_users["user"])]

    print("\nTrain label counts:")
    print(y_train["label"].value_counts())

    print("\nVal label counts:")
    print(y_val["label"].value_counts())

    return X_train, y_train, X_val, y_val


def compute_item_stats(XX_train: pd.DataFrame) -> dict:
    """Compute item-level statistics from training interactions ONLY.

    Pass the returned dict to build_features() for both train and test
    to prevent data leakage.
    """
    item_avg = XX_train.groupby("item")["rating"].mean().rename("item_avg_rating")
    item_pop = XX_train.groupby("item")["user"].count().rename("item_popularity")
    return {"item_avg_rating": item_avg, "item_popularity": item_pop}


def build_features(
    XX: pd.DataFrame,
    item_stats: dict,
    total_items: int=TOTAL_ITEMS,
) -> pd.DataFrame:
    """Build user-level features from raw interactions.

    Returns DataFrame with a 'user' column and all engineered features.
    """
    item_avg = item_stats["item_avg_rating"]
    item_pop = item_stats["item_popularity"]

    # Basic rating statistics
    stats = XX.groupby("user")["rating"].agg(
        rating_mean="mean",
        rating_std="std",
        rating_median="median",
        rating_min="min",
        rating_max="max",
        rating_count="count",
    )
    stats["rating_std"] = stats["rating_std"].fillna(0)
    stats["rating_range"] = stats["rating_max"] - stats["rating_min"]

    # Rating proportions + entropy
    rdist = XX.groupby(["user", "rating"]).size().unstack(fill_value=0)
    rdist = rdist.reindex(columns=RATING_RANGE, fill_value=0)
    rprops = rdist.div(rdist.sum(axis=1), axis=0)
    rprops.columns = [f"prop_rating_{i}" for i in RATING_RANGE]

    stats["rating_entropy"] = rprops.apply(
        lambda row: entropy(row.values[row.values > 0]), axis=1
    )
    stats = stats.join(rprops)

    # Extreme rating proportions
    stats["prop_extreme"] = rprops["prop_rating_0"] + rprops["prop_rating_5"]

    # Item coverage
    stats["unique_items_rated"] = XX.groupby("user")["item"].nunique()
    stats["item_coverage_ratio"] = stats["unique_items_rated"] / total_items

    # Item popularity (frozen from training)
    XX_pop = XX.merge(item_pop, left_on="item", right_index=True, how="left")
    XX_pop["item_popularity"] = XX_pop["item_popularity"].fillna(0)
    pop_f = XX_pop.groupby("user")["item_popularity"].agg(
        avg_item_popularity="mean",
        std_item_popularity="std",
    )
    pop_f["std_item_popularity"] = pop_f["std_item_popularity"].fillna(0)
    stats = stats.join(pop_f)

    # Deviation from item average (frozen from training)
    XX_dev = XX.merge(item_avg, left_on="item", right_index=True, how="left")
    global_train_mean = item_avg.mean()
    XX_dev["item_avg_rating"] = XX_dev["item_avg_rating"].fillna(global_train_mean)
    XX_dev["deviation"] = XX_dev["rating"] - XX_dev["item_avg_rating"]

    dev_f = XX_dev.groupby("user")["deviation"].agg(
        mean_deviation="mean",
        std_deviation="std",
        abs_mean_deviation=lambda x: np.mean(np.abs(x)),
    )
    dev_f["std_deviation"] = dev_f["std_deviation"].fillna(0)
    stats = stats.join(dev_f)

    # Average quality of items targeted (frozen from training)
    iqf = XX_dev.groupby("user")["item_avg_rating"].agg(
        avg_item_avg_rating="mean",
        std_item_avg_rating="std",
    )
    iqf["std_item_avg_rating"] = iqf["std_item_avg_rating"].fillna(0)
    stats = stats.join(iqf)

    return stats.reset_index()


def get_feature_columns(df: pd.DataFrame) -> list[str]:
    """Return feature column names (excludes 'user' and 'label')."""
    return [c for c in df.columns if c not in ("user", "label")]


In [2]:


# ============================================
# Features loaded from feature_pipeline in cell 1:
#   X_features (880 train users), y_labels
#   X_test (220 unseen test users), test_labels
# ============================================
def load_npz(path: str) -> tuple[pd.DataFrame, pd.DataFrame | None]:
    data = np.load(path)
    XX = pd.DataFrame(data["X"], columns=["user", "item", "rating"])
    yy = None
    if "y" in data:
        yy = pd.DataFrame(data["y"], columns=["user", "label"])
    return XX, yy


X, y = load_npz("data/training_batch_with_labels.npz")


# ============================================
# HOLD OUT 20% OF TRAINING FOR INTERNAL EVAL
# ============================================

train_users, heldout_users = train_test_split(
    y,
    test_size=0.2,
    stratify=y["label"],
    random_state=42
)

X_trainval, y_trainval, X_heldout, y_heldout = split_datasets(X, y, train_users, heldout_users)

# ============================================
# BUILD FEATURES
# ============================================
item_stats = compute_item_stats(X_trainval)

train_df = build_features(X_trainval, item_stats).merge(y_trainval, on="user")
heldout_df = build_features(X_heldout, item_stats).merge(y_heldout, on="user")

feature_cols = get_feature_columns(train_df)

X_train = train_df[feature_cols].values
y_train = train_df["label"].values

X_val = heldout_df[feature_cols].values
y_val = heldout_df["label"].values

# ============================================
# OPTIONAL TEST SET
# ============================================
X_test_raw, y_test_raw = load_npz("data/first_batch_with_labels.npz")
test_df = build_features(X_test_raw, item_stats)
X_test = test_df[feature_cols].values

y_test = None
if y_test_raw is not None:
    y_test = test_df[["user"]].merge(y_test_raw, on="user", how="left")["label"].values

# ============================================
# FEATURE SCALING
# ============================================
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# ============================================
# HELPER: build models
# ============================================
def make_xgb(scale_pos_weight, seed=42):
    return xgb.XGBClassifier(
        n_estimators=800,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.7,
        colsample_bylevel=0.7,
        min_child_weight=3,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=1.5,
        scale_pos_weight=scale_pos_weight,
        eval_metric="auc",
        early_stopping_rounds=50,
        random_state=seed,
        n_jobs=-1,
    )

def make_lgb(scale_pos_weight, seed=42):
    return lgb.LGBMClassifier(
        n_estimators=800,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.7,
        min_child_samples=20,
        reg_alpha=0.1,
        reg_lambda=1.5,
        scale_pos_weight=scale_pos_weight,
        random_state=seed,
        n_jobs=-1,
        verbose=-1,
    )

# ============================================
# CROSS VALIDATION
# ============================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

aucs_xgb, aucs_lgb, aucs_ens = [], [], []
oof_xgb = np.zeros(len(X_train))
oof_lgb = np.zeros(len(X_train))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_scaled, y_train)):
    X_tr, X_cv = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_tr, y_cv = y_train[train_idx], y_train[val_idx]

    spw = np.sum(y_tr == 0) / np.sum(y_tr == 1)

    xgb_model = make_xgb(spw)
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_cv, y_cv)], verbose=False)
    p_xgb = xgb_model.predict_proba(X_cv)[:, 1]
    oof_xgb[val_idx] = p_xgb

    lgb_model = make_lgb(spw)
    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_cv, y_cv)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
    )
    p_lgb = lgb_model.predict_proba(X_cv)[:, 1]
    oof_lgb[val_idx] = p_lgb

    p_ens = (rankdata(p_xgb) + rankdata(p_lgb)) / 2

    auc_xgb = roc_auc_score(y_cv, p_xgb)
    auc_lgb = roc_auc_score(y_cv, p_lgb)
    auc_ens = roc_auc_score(y_cv, p_ens)

    aucs_xgb.append(auc_xgb)
    aucs_lgb.append(auc_lgb)
    aucs_ens.append(auc_ens)

    print(f"Fold {fold+1}: XGB={auc_xgb:.4f} LGB={auc_lgb:.4f} Ensemble={auc_ens:.4f}")

oof_ens = (rankdata(oof_xgb) + rankdata(oof_lgb)) / 2
oof_auc = roc_auc_score(y_train, oof_ens)

print("\n=== CV Results ===")
print(f"XGBoost  AUC: {np.mean(aucs_xgb):.4f} ± {np.std(aucs_xgb):.4f}")
print(f"LightGBM AUC: {np.mean(aucs_lgb):.4f} ± {np.std(aucs_lgb):.4f}")
print(f"Ensemble AUC: {np.mean(aucs_ens):.4f} ± {np.std(aucs_ens):.4f}")
print(f"OOF Ensemble: {oof_auc:.4f}")

# ============================================
# FINAL MODELS
# ============================================
spw = np.sum(y_train == 0) / np.sum(y_train == 1)

scaler_final = RobustScaler()
X_all_scaled = scaler_final.fit_transform(X_train)

X_ft, X_es, y_ft, y_es = train_test_split(
    X_all_scaled, y_train, test_size=0.1, stratify=y_train, random_state=42
)

final_xgb = make_xgb(spw)
final_xgb.fit(X_ft, y_ft, eval_set=[(X_es, y_es)], verbose=False)

final_lgb = make_lgb(spw)
final_lgb.fit(
    X_ft, y_ft,
    eval_set=[(X_es, y_es)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
)

# ============================================
# EVALUATE ON HELD-OUT SET
# ============================================
X_heldout_final = scaler_final.transform(X_val)
p_xgb_h = final_xgb.predict_proba(X_heldout_final)[:, 1]
p_lgb_h = final_lgb.predict_proba(X_heldout_final)[:, 1]
y_score_h = (rankdata(p_xgb_h) + rankdata(p_lgb_h)) / 2

print(f"\nXGB + LGB Ensemble (internal held-out):")
print(f"AUC: {roc_auc_score(y_val, y_score_h):.4f}")

# ============================================
# EVALUATE ON TEST SET IF LABELS EXIST
# ============================================
X_test_scaled = scaler_final.transform(X_test)
p_xgb_test = final_xgb.predict_proba(X_test_scaled)[:, 1]
p_lgb_test = final_lgb.predict_proba(X_test_scaled)[:, 1]
y_score_test = (rankdata(p_xgb_test) + rankdata(p_lgb_test)) / 2

if y_test is not None:
    oof_ens_norm = (oof_ens - oof_ens.min()) / (oof_ens.max() - oof_ens.min() + 1e-9)
    thresholds = np.linspace(0.01, 0.99, 500)
    best_f1, best_threshold = 0, 0.5

    for t in thresholds:
        preds_t = (oof_ens_norm >= t).astype(int)
        tp = np.sum((preds_t == 1) & (y_train == 1))
        fp = np.sum((preds_t == 1) & (y_train == 0))
        fn = np.sum((preds_t == 0) & (y_train == 1))
        prec = tp / (tp + fp + 1e-9)
        rec = tp / (tp + fn + 1e-9)
        f1_t = 2 * prec * rec / (prec + rec + 1e-9)
        if f1_t > best_f1:
            best_f1, best_threshold = f1_t, t

    y_score_test_norm = (y_score_test - y_score_test.min()) / (y_score_test.max() - y_score_test.min() + 1e-9)
    y_pred_test = (y_score_test_norm >= best_threshold).astype(int)

    print(f"\nXGB + LGB Ensemble (local test set):")
    print(f"AUC:       {roc_auc_score(y_test, y_score_test):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred_test, zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred_test, zero_division=0):.4f}")
    print(f"F1 Score:  {f1_score(y_test, y_pred_test, zero_division=0):.4f}")

# ============================================
# PREDICT
# ============================================
p_xgb_test = final_xgb.predict_proba(X_test_scaled)[:, 1]
p_lgb_test = final_lgb.predict_proba(X_test_scaled)[:, 1]

# Rank-based ensemble (same as your training pipeline)
y_score_test = (rankdata(p_xgb_test) + rankdata(p_lgb_test)) / 2

# Normalize scores to [0, 1]
y_score_test_norm = (y_score_test - y_score_test.min()) / (
    y_score_test.max() - y_score_test.min() + 1e-9
)

# ============================================
# CHOOSE THRESHOLD FROM OOF / TRAINING
# ============================================
oof_ens_norm = (oof_ens - oof_ens.min()) / (oof_ens.max() - oof_ens.min() + 1e-9)

thresholds = np.linspace(0.01, 0.99, 500)
best_f1, best_threshold = 0, 0.5

for t in thresholds:
    preds_t = (oof_ens_norm >= t).astype(int)
    f1_t = f1_score(y_train, preds_t, zero_division=0)
    if f1_t > best_f1:
        best_f1 = f1_t
        best_threshold = t

# Apply threshold to current labelled test set
y_pred_test = (y_score_test_norm >= best_threshold).astype(int)

# ============================================
# METRICS
# ============================================
print("\n=== Current Labelled Test Set Metrics ===")
print(f"AUC:       {roc_auc_score(y_test, y_score_test_norm):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_test, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_test, zero_division=0):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_test, zero_division=0):.4f}")
print(f"Threshold: {best_threshold:.4f}")

# Optional: confusion matrix
cm = confusion_matrix(y_test, y_pred_test)
print("\nConfusion Matrix:")
print(cm)


Train label counts:
label
0    800
1     80
Name: count, dtype: int64

Val label counts:
label
0    200
1     20
Name: count, dtype: int64


/Users/tori/Documents/OFFICES/SCHOOL/Y3S2/Machine_Learning_421_SMU/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 1: XGB=0.9277 LGB=0.9254 Ensemble=0.9320


/Users/tori/Documents/OFFICES/SCHOOL/Y3S2/Machine_Learning_421_SMU/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 2: XGB=0.7488 LGB=0.7109 Ensemble=0.7297


...

Fold 4: XGB=0.9215 LGB=0.9047 Ensemble=0.9150


/Users/tori/Documents/OFFICES/SCHOOL/Y3S2/Machine_Learning_421_SMU/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 5: XGB=0.9590 LGB=0.9492 Ensemble=0.9604

=== CV Results ===
XGBoost  AUC: 0.9002 ± 0.0768
LightGBM AUC: 0.8854 ± 0.0884
Ensemble AUC: 0.8958 ± 0.0844
OOF Ensemble: 0.8974


/Users/tori/Documents/OFFICES/SCHOOL/Y3S2/Machine_Learning_421_SMU/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/tori/Documents/OFFICES/SCHOOL/Y3S2/Machine_Learning_421_SMU/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/tori/Documents/OFFICES/SCHOOL/Y3S2/Machine_Learning_421_SMU/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



XGB + LGB Ensemble (internal held-out):
AUC: 0.9533

XGB + LGB Ensemble (local test set):
AUC:       0.8993
Precision: 0.6707
Recall:    0.5500
F1 Score:  0.6044

=== Current Labelled Test Set Metrics ===
AUC:       0.8993
Precision: 0.6707
Recall:    0.5500
F1 Score:  0.6044
Threshold: 0.9213

Confusion Matrix:
[[973  27]
 [ 45  55]]
